# Model 2 — Variance Threshold + Random Forest (GLOBAL — All 9 Devices)

Removes features whose values barely change across samples (near-zero variance).
| Model | Features Used | Classifier |
|---|---|---|
| Model 1 | All 115 features (baseline) | Random Forest |
| Model 2 | Variance Threshold subset | Random Forest |
| Model 3 | Pearson Correlation subset | Random Forest |
| Model 4 | Mutual Information top-40 | Random Forest |


**Fixed version:** memory-safe sampling, float32, shared cached training sample (identical across all 4 models), progress prints, faster evaluation, and a `QUICK_TEST` switch. Run the fixed Model 1 notebook first so the shared sample cache exists.

In [ ]:
from pathlib import Path
import json, time, warnings, gc
import numpy as np
import pandas as pd
import joblib

from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import LabelEncoder
from sklearn.feature_selection import VarianceThreshold
from sklearn.metrics import (
    accuracy_score, precision_recall_fscore_support,
    classification_report, confusion_matrix
)

warnings.filterwarnings('ignore')

PROJECT_ROOT = Path(r'c:\Users\kulsu\Downloads\detection+of+iot+botnet+attacks+n+baiot')
PREPROCESSED = PROJECT_ROOT / 'preprocessed'

ALL_DEVICES = [
    'Danmini_Doorbell', 'Ecobee_Thermostat', 'Ennio_Doorbell',
    'Philips_B120N10_Baby_Monitor', 'Provision_PT_737E_Security_Camera',
    'Provision_PT_838_Security_Camera', 'Samsung_SNH_1011_N_Webcam',
    'SimpleHome_XCS7_1002_WHT_Security_Camera',
    'SimpleHome_XCS7_1003_WHT_Security_Camera',
]

# ---- QUICK TEST: True = 1 device only (minutes) to verify the pipeline ----
QUICK_TEST = False
DEVICES = ALL_DEVICES[:1] if QUICK_TEST else ALL_DEVICES

TRAIN_CHUNK_SIZE = 100_000     # lower to 50_000 if you have <= 8 GB RAM
TEST_CHUNK_SIZE  = 200_000
MAX_PER_CLASS_PER_DEVICE = 2_000
RANDOM_STATE = 42
USE_CACHE = True
VARIANCE_THRESHOLD = 0.01

# IDENTICAL for all 4 models (fair comparison)
RF_PARAMS = dict(
    n_estimators=200, max_depth=25, min_samples_leaf=2, max_features='sqrt',
    class_weight='balanced_subsample', random_state=RANDOM_STATE, n_jobs=-1
)

sample_header = pd.read_csv(PREPROCESSED / ALL_DEVICES[0] / 'train.csv', nrows=0)
all_feature_cols = [c for c in sample_header.columns if c not in {'label', 'label_id'}]
FEAT_DTYPES = {c: 'float32' for c in all_feature_cols}
print(f'Devices: {len(DEVICES)} (QUICK_TEST={QUICK_TEST}) | Features: {len(all_feature_cols)}')

---
## 1. Load Training Data (shared sample)

All 4 models train on the **same sample**. This cell loads the cached sample created by the fixed Model 1 notebook (`global_train_sample.joblib`). If it doesn't exist yet, it builds it with the exact same memory-safe sampler and seeds, then saves it, so every model sees identical data.

In [ ]:
def sample_device_train(device_name, feat_cols, max_per_class, seed=42):
    """Memory-safe sampler (same as fixed Model 1): caps each class DURING reading."""
    rng = np.random.default_rng(seed)
    train_path = PREPROCESSED / device_name / 'train.csv'
    kept, kept_n = {}, {}

    for i, chunk in enumerate(pd.read_csv(train_path, chunksize=TRAIN_CHUNK_SIZE,
                                          usecols=feat_cols + ['label'],
                                          dtype=FEAT_DTYPES)):
        chunk = chunk.dropna(subset=['label'])
        for lbl, g in chunk.groupby('label'):
            if len(g) > max_per_class:
                g = g.sample(n=max_per_class, random_state=int(rng.integers(1_000_000_000)))
            kept.setdefault(lbl, []).append(g)
            kept_n[lbl] = kept_n.get(lbl, 0) + len(g)
            if kept_n[lbl] > 3 * max_per_class:
                merged = pd.concat(kept[lbl], ignore_index=True)
                merged = merged.sample(n=max_per_class, random_state=int(rng.integers(1_000_000_000)))
                kept[lbl] = [merged]
                kept_n[lbl] = len(merged)
        if i % 10 == 0:
            print(f'      chunk {i} read', flush=True)

    sampled = []
    for lbl in sorted(kept):
        df_cls = pd.concat(kept[lbl], ignore_index=True)
        if len(df_cls) > max_per_class:
            df_cls = df_cls.sample(n=max_per_class, random_state=seed)
        sampled.append(df_cls)
    del kept
    gc.collect()
    return pd.concat(sampled, ignore_index=True)


cache_file = PREPROCESSED / ('global_train_sample_quick.joblib' if QUICK_TEST
                             else 'global_train_sample.joblib')

t0 = time.time()
if USE_CACHE and cache_file.exists():
    print(f'Loading shared cached sample: {cache_file.name}')
    df_train_global = joblib.load(cache_file)
else:
    parts = []
    for n, dev in enumerate(DEVICES, 1):
        t_dev = time.time()
        print(f'  [{n}/{len(DEVICES)}] Loading {dev}...', flush=True)
        part = sample_device_train(dev, all_feature_cols, MAX_PER_CLASS_PER_DEVICE, RANDOM_STATE)
        part['device'] = dev
        print(f'    -> {len(part):,} rows | Classes: {part["label"].nunique()} | {time.time()-t_dev:.0f}s', flush=True)
        parts.append(part)
    df_train_global = pd.concat(parts, ignore_index=True)
    del parts; gc.collect()
    if USE_CACHE:
        joblib.dump(df_train_global, cache_file)
        print(f'Cached sample to {cache_file.name}')

print(f'\nGlobal training set: {len(df_train_global):,} rows in {time.time()-t0:.1f}s')

le = LabelEncoder()
le.fit(df_train_global['label'])
global_label_map = {label: int(idx) for idx, label in enumerate(le.classes_)}

X_train_full = df_train_global[all_feature_cols].apply(pd.to_numeric, errors='coerce')
X_train_full = X_train_full.replace([np.inf, -np.inf], np.nan).fillna(0).astype('float32')
y_train = le.transform(df_train_global['label'])
print(f'X_train_full: {X_train_full.shape} | Classes: {len(np.unique(y_train))}')

---
## 2. Apply Variance Threshold Feature Selection

In [ ]:
selector = VarianceThreshold(threshold=VARIANCE_THRESHOLD)
X_train_sel = selector.fit_transform(X_train_full)

mask = selector.get_support()
selected_features = [all_feature_cols[i] for i, k in enumerate(mask) if k]
dropped_features  = [all_feature_cols[i] for i, k in enumerate(mask) if not k]

print('--- Variance Threshold Results ---')
print(f'  Original : {len(all_feature_cols)}')
print(f'  Selected : {len(selected_features)}')
print(f'  Dropped  : {len(dropped_features)} ({len(dropped_features)/len(all_feature_cols)*100:.1f}%)')
if dropped_features:
    print('\n  Dropped features:')
    for f in dropped_features:
        print(f'    {f:40s} (var = {X_train_full[f].var():.6f})')

X_train = pd.DataFrame(X_train_sel, columns=selected_features)

---
## 3. Train Random Forest

In [ ]:
model = RandomForestClassifier(**RF_PARAMS)
print(f'Training RF with {len(selected_features)} features on {len(X_train):,} rows...', flush=True)
t0 = time.time()
model.fit(X_train, y_train)
train_time = time.time() - t0
print(f'Done in {train_time:.1f}s')

---
## 4. Evaluate on All Devices' Test Sets

In [ ]:
def evaluate_all_devices(model, le, feat_cols, devices):
    y_true_all, y_pred_all = [], []
    device_results = {}
    total_inf = 0.0
    class_set = set(le.classes_)
    feat_dtypes = {c: 'float32' for c in feat_cols}

    for n, dev in enumerate(devices, 1):
        test_path = PREPROCESSED / dev / 'test.csv'
        yt_dev, yp_dev = [], []
        t_dev = time.time()
        for i, chunk in enumerate(pd.read_csv(test_path, chunksize=TEST_CHUNK_SIZE,
                                              usecols=feat_cols + ['label'],
                                              dtype=feat_dtypes)):
            chunk = chunk[chunk['label'].isin(class_set)]
            if len(chunk) == 0:
                continue
            yt = le.transform(chunk['label'])
            X = chunk[feat_cols].replace([np.inf, -np.inf], np.nan).fillna(0).astype('float32')
            t0 = time.time()
            yp = model.predict(X)
            total_inf += time.time() - t0
            yt_dev.append(yt); yp_dev.append(yp)
            print(f'    [{n}/{len(devices)}] {dev} chunk {i} done ({time.time()-t_dev:.0f}s)', flush=True)
        yt = np.concatenate(yt_dev); yp = np.concatenate(yp_dev)
        acc = accuracy_score(yt, yp)
        device_results[dev] = {'accuracy': float(acc), 'rows': int(len(yt))}
        print(f'  {dev:45s}  acc={acc:.4f}  rows={len(yt):>8,}', flush=True)
        y_true_all.append(yt); y_pred_all.append(yp)

    y_true = np.concatenate(y_true_all); y_pred = np.concatenate(y_pred_all)
    labels = sorted(set(y_true) | set(y_pred))
    p_m, r_m, f_m, _ = precision_recall_fscore_support(y_true, y_pred, average='macro', zero_division=0)
    p_w, r_w, f_w, _ = precision_recall_fscore_support(y_true, y_pred, average='weighted', zero_division=0)
    bi = global_label_map.get('benign', 0)
    bm = (y_true == bi)
    fpr = 1.0 - accuracy_score(y_true[bm], y_pred[bm]) if bm.sum() > 0 else None
    tn = [le.classes_[i] for i in labels]
    return {
        'accuracy': float(accuracy_score(y_true, y_pred)),
        'macro_precision': float(p_m), 'macro_recall': float(r_m), 'macro_f1': float(f_m),
        'weighted_precision': float(p_w), 'weighted_recall': float(r_w), 'weighted_f1': float(f_w),
        'false_positive_rate': float(fpr) if fpr is not None else None,
        'total_inference_time': total_inf,
        'avg_inference_per_sample_ms': float(total_inf / len(y_true) * 1000),
        'test_rows': int(len(y_true)), 'labels': [int(l) for l in labels],
        'confusion_matrix': confusion_matrix(y_true, y_pred, labels=labels),
        'classification_report': classification_report(y_true, y_pred, labels=labels, target_names=tn, zero_division=0),
        'per_device': device_results
    }

print(f'Evaluating on {len(DEVICES)} device test set(s)...', flush=True)
t0 = time.time()
metrics = evaluate_all_devices(model, le, selected_features, DEVICES)
eval_time = time.time() - t0

print(f"\n{'='*55}")
print(f'  MODEL 2: VARIANCE THRESHOLD — GLOBAL')
print(f"{'='*55}")
print(f'  Features kept   : {len(selected_features)} / {len(all_feature_cols)}')
print(f'  Accuracy        : {metrics["accuracy"]:.4f}')
print(f'  Macro Precision : {metrics["macro_precision"]:.4f}')
print(f'  Macro Recall    : {metrics["macro_recall"]:.4f}')
print(f'  Macro F1-Score  : {metrics["macro_f1"]:.4f}')
fv = metrics["false_positive_rate"]
print(f'  FPR (benign)    : {fv:.4f}' if fv is not None else '  FPR (benign)    : n/a')
print(f'  Inference time  : {metrics["total_inference_time"]:.2f}s')
print(f'  Test rows       : {metrics["test_rows"]:,}')
print(f"{'='*55}")
print(f'\n{metrics["classification_report"]}')

---
## 5. Save Outputs

In [ ]:
output_dir = PREPROCESSED / 'global_models' / ('Model2_VarianceThreshold_QUICKTEST' if QUICK_TEST else 'Model2_VarianceThreshold')
output_dir.mkdir(parents=True, exist_ok=True)

joblib.dump(model, output_dir / 'random_forest.joblib')
joblib.dump(le, output_dir / 'label_encoder.joblib')
joblib.dump(selector, output_dir / 'variance_threshold_selector.joblib')

with open(output_dir / 'selected_features.json', 'w') as f:
    json.dump({'method': 'variance_threshold', 'threshold': VARIANCE_THRESHOLD, 'original_count': len(all_feature_cols), 'selected_count': len(selected_features), 'dropped_count': len(dropped_features), 'selected': selected_features, 'dropped': dropped_features}, f, indent=2)

ms = {k: v for k, v in metrics.items() if k not in {'confusion_matrix', 'classification_report'}}
ms.update({'model': 'Model2_VarianceThreshold', 'scope': 'global_all_devices' if not QUICK_TEST else 'quick_test_1_device',
           'devices': DEVICES, 'feature_selection': 'variance_threshold',
           'feature_count': len(selected_features), 'training_rows': int(len(y_train)),
           'training_time_s': round(train_time, 2), 'evaluation_time_s': round(eval_time, 2),
           'rf_params': {k: str(v) for k, v in RF_PARAMS.items()}, 'threshold': VARIANCE_THRESHOLD})
with open(output_dir / 'metrics.json', 'w') as f:
    json.dump(ms, f, indent=2)

tn = [le.classes_[i] for i in metrics['labels']]
pd.DataFrame(metrics['confusion_matrix'], index=[f'actual_{n}' for n in tn],
             columns=[f'pred_{n}' for n in tn]).to_csv(output_dir / 'confusion_matrix.csv')
with open(output_dir / 'classification_report.txt', 'w') as f:
    f.write(metrics['classification_report'])
pd.DataFrame([{'Device': d, 'Accuracy': v['accuracy'], 'Test_Rows': v['rows']}
              for d, v in metrics['per_device'].items()]).to_csv(output_dir / 'per_device_accuracy.csv', index=False)

print(f'Saved to: {output_dir}')
for f in sorted(output_dir.iterdir()): print(f'   {f.name}')
print('\nNext: Run Model 3 (Pearson Correlation) notebook')